## Prior-Guided Graph Learning Block 可解释性分析

## 对 ADSSTNet 中 **Prior-Guided Graph Learning Block** 的初始化邻接矩阵 **A0** 进行可视化：

1. **参数配置**：与训练脚本 `ADSSTNet-Ablation-Experiment.sh` 保持一致，独立初始化（参考 `Pearson_Connectivity_Analysis_for_.ipynb` 的风格）
2. **A0 计算**：复现 `exp_supervised._build_model` 中的 `compute_pearson_A0_subject_mean` 逻辑
3. **5 折可视化**：对应 `itrs=5` 的 Monte Carlo 交叉验证，每折使用不同 `seed` 划分训练集并绘制 A0 热力图

In [1]:
import os
import sys
import random
from argparse import Namespace
from pathlib import Path

import matplotlib.cm as cm
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

# 将项目根目录加入 sys.path，以便导入 data_provider / utils
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'run.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_provider.data_factory import data_dict
from utils.tools import compute_pearson_A0_subject_mean

In [2]:
# ========== 路径与可视化配置==========
save_path = PROJECT_ROOT / "figs" / "Prior-Guided Graph Learning"
OUTPUT_DIR = save_path / "A0_explainability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ========== 随机种子 交叉验证 ==========
BASE_SEED = 38          # run.py 中 --seed
ITRS = 5                # run.py 中 --itrs（5 折 Monte Carlo CV）

# 20 通道标准 10-20 名称（与 CAEEG channel_orders 一致）
CH_NAMES = [
    'Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'Cz', 'C3', 'C4',
    'T7', 'T8', 'Pz', 'P3', 'P4', 'P7', 'P8', 'Oz', 'O1', 'O2',
]

# 绘图样式
font_path = PROJECT_ROOT / "figs/fonts/TIMES.TTF"
font_manager.fontManager.addfont(font_path)
PLOT_FONT = 'Times New Roman'
FONT_TITLE = 12
FONT_AXIS_LABEL = 11
FONT_TICK = 9

# A0 热力图配色；修改 CMAP_A0_NAME 即可切换（0 值格点不绘制，见 _mask_zero_and_diag）
CMAP_A0_OPTIONS = {
    'ylorrd':  'YlOrRd',   # 黄-橙-红（默认）
    'oranges': 'Oranges',  # 橙色系
    'reds':    'Reds',     # 红色系
    'blues':   'Blues',    # 蓝色系
    'greens':  'Greens',   # 绿色系
    'purples': 'Purples',  # 紫色系
    'cividis': 'Cividis',  # 黄-蓝（色盲友好）
    'magma':   'Magma',    # 紫-黄
}
CMAP_A0_NAME = 'ylorrd'
CMAP_ADJ_MINUS_A0 = 'RdBu_r'  # adjacency − A0（可正可负）

plt.rcParams.update({'font.family': PLOT_FONT, 'font.serif': [PLOT_FONT]})

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")
print(f"CMAP_A0_NAME = {CMAP_A0_NAME}  (可选: {list(CMAP_A0_OPTIONS)})")

PROJECT_ROOT = /root/autodl-tmp/ADSST-Net
OUTPUT_DIR   = /root/autodl-tmp/ADSST-Net/figs/Prior-Guided Graph Learning/A0_explainability
CMAP_A0_NAME = ylorrd  (可选: ['ylorrd', 'oranges', 'reds', 'blues', 'greens', 'purples', 'cividis', 'magma'])


### 逐折计算初始化 A0 并绘制热力图

与 `run.py` 一致：第 `k` 折（`k=0..4`）使用 `seed = BASE_SEED + k + 1`，
在对应训练集上调用 `compute_pearson_A0_subject_mean` 得到初始化 A0。

In [3]:
# 数据加载（划分 TRAIN 集）所需
args = Namespace(
    seed=BASE_SEED,
    data='CAEEG',
    root_path=str((PROJECT_ROOT / 'dataset/CAEEG3_1.0s_0.0_EO_Aug').resolve()) + '/',
    selected_classes=[0, 1, 2],
    cross_val='mccv',
    topk_A0=4,
)

# compute_pearson_A0_subject_mean 参数（与 exp_supervised._build_model 一致）
A0_KWARGS = dict(
    use_abs=True,
    topk=args.topk_A0,
    sym_mode='max',
    eps=1e-8,
    row_normalize=True,
    clip_negative=True,
    dataset_name=args.data,
)

print(f"data={args.data}, root_path={args.root_path}")
print(f"selected_classes={args.selected_classes}, topk_A0={args.topk_A0}")
print(f"BASE_SEED={BASE_SEED}, ITRS={ITRS}")

data=CAEEG, root_path=/root/autodl-tmp/ADMamba/dataset/USTBMCI3_1.0s_0.0_EO_Aug/
selected_classes=[0, 1, 2], topk_A0=4
BASE_SEED=38, ITRS=5


In [4]:
def load_train_dataset(seed: int):
    """按指定 seed 加载 TRAIN 数据集（与 run.py / exp_supervised 一致）。"""
    fold_args = Namespace(**vars(args))
    fold_args.seed = seed
    random.seed(seed)
    np.random.seed(seed)
    return data_dict[fold_args.data](args=fold_args, flag='TRAIN', log=None)


def compute_init_A0(train_data, enc_in: int) -> np.ndarray:
    """复现 exp_supervised._build_model 中的 A0 初始化。"""
    return compute_pearson_A0_subject_mean(train_data, enc_in=enc_in, **A0_KWARGS)


def _a0_cmap_for_plot():
    if CMAP_A0_NAME not in CMAP_A0_OPTIONS:
        raise ValueError(f"未知配色 '{CMAP_A0_NAME}'，可选: {list(CMAP_A0_OPTIONS)}")
    return cm.get_cmap(CMAP_A0_OPTIONS[CMAP_A0_NAME])


ADJ_ZERO_ATOL = 1e-12  # 视为「无边」的阈值（off-diagonal |w| <= atol）


def _prep_adj_matrix(mat: np.ndarray, *, diverging: bool = False) -> np.ma.MaskedArray:
    """对角线与无边(off-diagonal 近零)置为掩蔽，imshow 显示为白色。"""
    m = mat.astype(float, copy=True)
    n = m.shape[0]
    np.fill_diagonal(m, np.nan)
    off_diag = ~np.eye(n, dtype=bool)
    no_edge = off_diag & np.isclose(m, 0.0, atol=ADJ_ZERO_ATOL, rtol=0.0)
    m[no_edge] = np.nan
    return np.ma.masked_invalid(m)


def _mask_zero_and_diag(mat: np.ndarray) -> np.ndarray:
    """对角线与 0 值置为 NaN，imshow 不绘制这些格点。"""
    return _prep_adj_matrix(mat, diverging=False)


def _cmap_for_adj_plot(cmap_name: str | None = None):
    if cmap_name is not None:
        cmap = plt.colormaps[cmap_name]
    else:
        cmap = _a0_cmap_for_plot()
    if hasattr(cmap, 'resampled'):
        cmap = cmap.resampled(256)
    cmap = cmap.copy()
    cmap.set_bad('#ffffff')
    return cmap


def _set_subplot_white_bg(ax):
    """子图面板与掩蔽格点均显示为白色背景。"""
    ax.set_facecolor('white')
    ax.patch.set_facecolor('white')


def _imshow_adj_matrix(ax, mat, *, cmap, vmin: float, vmax: float):
    """在 ax 上绘制邻接矩阵热力图（掩蔽/无边格点为白色）。"""
    _set_subplot_white_bg(ax)
    m = mat if isinstance(mat, np.ma.MaskedArray) else _prep_adj_matrix(mat)
    n = m.shape[0]
    edges = np.arange(n + 1) - 0.5
    mesh = ax.pcolormesh(
        edges, edges, m,
        cmap=cmap, vmin=vmin, vmax=vmax,
        shading='flat', edgecolors='none', antialiased=False,
    )
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)
    return mesh


def _add_channel_grid(ax, n_ch: int, *, color: str = '#d8d8d8', linewidth: float = 0.35):
    """在热力图上方绘制通道分隔网格（比 ax.grid 更适合 SVG 导出）。"""
    from matplotlib.collections import LineCollection

    edges = np.arange(-0.5, n_ch, 1.0)
    segments = []
    for x in edges:
        segments.append([(x, -0.5), (x, n_ch - 0.5)])
    for y in edges:
        segments.append([(-0.5, y), (n_ch - 0.5, y)])
    lc = LineCollection(
        segments, colors=color, linewidths=linewidth, zorder=3,
        antialiaseds=(0,),
    )
    ax.add_collection(lc)


def _vmin_vmax_from_mats(mats, *, diverging: bool = False) -> tuple[float, float]:
    """根据有限元素统计 colorbar 范围；发散图对称于 0。"""
    parts = [
        m.compressed() if isinstance(m, np.ma.MaskedArray) else m[np.isfinite(m)]
        for m in mats
    ]
    vals = np.concatenate(parts) if parts else np.array([])
    if vals.size == 0:
        return (-1.0, 1.0) if diverging else (0.0, 1.0)
    vmin = float(np.min(vals))
    vmax = float(np.max(vals))
    if diverging:
        vabs = max(abs(vmin), abs(vmax), 1e-6)
        return -vabs, vabs
    if vmin >= vmax:
        vmax = vmin + 1e-6
    return vmin, vmax


def plot_A0_heatmap_grid(
    A0_list,
    fold_labels,
    save_path: Path,
    suptitle: str,
    *,
    cmap_name: str | None = None,
    diverging: bool = False,
    value_label: str = 'Edge weight',
):
    """将多折矩阵绘制在同一张图的子图中便于对比。"""
    n_folds = len(A0_list)
    n_ch = len(CH_NAMES)

    mats = [_prep_adj_matrix(A0, diverging=diverging) for A0 in A0_list]
    vmin, vmax = _vmin_vmax_from_mats(mats, diverging=diverging)
    cmap = _cmap_for_adj_plot(cmap_name)

    fig = plt.figure(figsize=(4.0 * n_folds + 0.8, 4.5), facecolor='white')
    gs = fig.add_gridspec(1, n_folds + 1, width_ratios=[1] * n_folds + [0.04], wspace=0.3)
    axes = [fig.add_subplot(gs[0, i]) for i in range(n_folds)]
    cax = fig.add_subplot(gs[0, n_folds])

    im = None
    for ax, mat, label in zip(axes, mats, fold_labels):
        im = _imshow_adj_matrix(ax, mat, cmap=cmap, vmin=vmin, vmax=vmax)
        _add_channel_grid(ax, n_ch)
        ax.set_title(label, fontsize=FONT_TITLE, fontweight='bold')
        ax.set_xticks(range(n_ch))
        ax.set_yticks(range(n_ch))
        ax.set_xticklabels(CH_NAMES, rotation=90, ha='center', fontsize=7)
        ax.set_yticklabels(CH_NAMES, fontsize=7)
        ax.tick_params(axis='both', which='both', length=0, width=0)
        ax.grid(False)
        ax.set_aspect('equal')

    _set_subplot_white_bg(cax)
    fig.suptitle(suptitle, fontsize=FONT_TITLE + 1, fontweight='bold', y=1.02)
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label(value_label, fontsize=FONT_AXIS_LABEL)
    cbar.ax.tick_params(labelsize=FONT_TICK)
    save_path = Path(save_path)
    if save_path.suffix.lower() != '.svg':
        save_path = save_path.with_suffix('.svg')
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, format='svg', bbox_inches='tight', facecolor='white')
    plt.close(fig)


import networkx as nx
from nilearn import plotting

# EEG 通道 MNI 坐标（与 CH_NAMES 顺序一致）
EEG_CHANNEL_COORDS = np.array([
    [-19.,  59.,   -2.],    # Fp1
    [ 19.,  59.,   -2.],    # Fp2
    [  0.,  18.,   48.],    # Fz
    [-21.,  22.,   24.],    # F3
    [ 21.,  22.,   24.],    # F4
    [-42.,  30.,   -2.],    # F7
    [ 42.,  30.,   -2.],    # F8
    [  0.,  -20.,  70.],    # Cz
    [-30.,  -20.,  48.],    # C3
    [ 30.,  -20.,  48.],    # C4
    [-60.,  -20.,  -2.],    # T7
    [ 60.,  -20.,  -2.],    # T8
    [  0.,  -50.,  48.],    # Pz
    [-21.,  -54.,  24.],    # P3
    [ 21.,  -54.,  24.],    # P4
    [-42.,  -62.,  -2.],    # P7
    [ 42.,  -62.,  -2.],    # P8
    [  0.,  -92.,  -2.],    # Oz
    [-19.,  -88.,  -2.],    # O1
    [ 19.,  -88.,  -2.],    # O2
])


def symmetrize_A0_for_plot(A: np.ndarray) -> np.ndarray:
    """行归一化后的 A0 一般不对称，connectome 绘图前先对称化。"""
    A = np.asarray(A, dtype=np.float64)
    A_plot = 0.5 * (A + A.T)
    np.fill_diagonal(A_plot, 0.0)
    return A_plot


def plot_A0_connectome(
    A0: np.ndarray,
    *,
    save_path: Path | str,
    ch_names=None,
    coords=None,
    linewidth: float = 1.5,
    dpi: int = 300,
):
    """
    绘制对称化后的 A0 3D connectome 并保存为图片。
    行归一化后的 A0 不对称，绘图前先做 (A + A.T) / 2 对称化。
    显示 A0 中所有非零连接（不做强度阈值过滤）。
    颜色映射：A0 最小值 -> 色棒最浅色，A0 最大值 -> 色棒最深色。
    """
    ch_names = list(ch_names or CH_NAMES)
    coords = np.asarray(coords if coords is not None else EEG_CHANNEL_COORDS, dtype=np.float64)
    n_nodes = len(ch_names)
    if A0.shape != (n_nodes, n_nodes):
        raise ValueError(f'A0 shape {A0.shape} 与 n_nodes={n_nodes} 不匹配')

    A_plot = symmetrize_A0_for_plot(A0)

    G = nx.from_numpy_array(np.abs(A_plot))
    degrees = dict(nx.degree(G))
    node_sizes = np.array([degrees.get(i, 0) + 30 for i in range(n_nodes)])

    off_diag = A_plot.copy()
    np.fill_diagonal(off_diag, 0.0)
    edge_vals = off_diag[off_diag > 0]
    if edge_vals.size:
        vmin = float(edge_vals.min())
        vmax = float(edge_vals.max())
    else:
        vmin, vmax = 0.0, 1.0
    if vmin >= vmax:
        vmax = vmin + 1e-6
    edge_cmap = CMAP_A0_OPTIONS.get(CMAP_A0_NAME, 'YlOrRd')
    save_filename = Path(save_path).with_suffix('.png')

    display = plotting.plot_connectome(
        adjacency_matrix=A_plot,
        node_coords=coords,
        node_color='auto',
        node_size=node_sizes,
        edge_cmap=edge_cmap,
        edge_threshold=0,
        edge_kwargs={'linewidth': linewidth},
        edge_vmin=vmin,
        edge_vmax=vmax,
        colorbar=True,
    )
    display.savefig(save_filename, dpi=dpi, bbox_inches='tight')
    display.close()


### 计算初始化 A0 矩阵

In [5]:
A0_by_fold = {}
fold_meta = []

for fold_idx in range(ITRS):
    # 与 run.py 一致：每折开始前 seed += 1
    fold_seed = BASE_SEED + fold_idx + 1
    fold_tag = f'itr_{fold_idx:03d}'

    train_data = load_train_dataset(fold_seed)
    enc_in = train_data.X.shape[-1]

    A0 = compute_init_A0(train_data, enc_in=enc_in)
    A0_by_fold[fold_tag] = A0
    fold_meta.append({'fold': fold_idx + 1, 'seed': fold_seed, 'tag': fold_tag, 'n_train': len(train_data.y)})

    off_diag = A0.copy()
    np.fill_diagonal(off_diag, 0.0)
    n_edges = int(np.count_nonzero(off_diag))
    print(
        f"{fold_tag} (seed={fold_seed}): "
        f"samples={len(train_data.y)}, edges={n_edges}, "
        f"max={off_diag.max():.4f}, mean(nonzero)={off_diag[off_diag > 0].mean():.4f}, "
    )

train ids: [2, 5, 6, 7, 8, 11, 13, 14, 15, 17, 19, 20, 22, 27, 29, 31, 32, 33, 34, 35, 36, 38, 39, 41, 42, 44, 46, 48, 49, 50, 52, 54, 55, 56, 57, 59, 60, 61, 64, 67, 69, 70, 74, 75, 77, 78, 79, 80, 81, 82, 84, 88, 90, 91, 92, 93, 96, 97, 98, 99, 100, 101, 102, 104, 105, 106, 107, 108, 109, 111] (数量: 70)
itr_000 (seed=39): samples=12424, edges=106, max=0.3142, mean(nonzero)=0.1887, 
train ids: [1, 2, 3, 4, 7, 9, 10, 11, 14, 16, 17, 18, 20, 23, 26, 27, 28, 29, 33, 34, 36, 37, 39, 40, 41, 42, 45, 46, 47, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 62, 64, 65, 66, 67, 68, 69, 70, 72, 73, 77, 79, 80, 81, 82, 83, 84, 85, 86, 90, 95, 96, 98, 99, 101, 103, 105, 107, 108, 110, 111] (数量: 70)
itr_001 (seed=40): samples=12437, edges=104, max=0.3187, mean(nonzero)=0.1923, 
train ids: [1, 2, 6, 7, 8, 11, 12, 13, 15, 17, 18, 20, 21, 23, 25, 26, 27, 30, 33, 34, 35, 39, 40, 41, 42, 43, 44, 45, 46, 51, 53, 56, 57, 58, 59, 63, 65, 66, 67, 68, 69, 70, 71, 72, 73, 75, 77, 78, 81, 82, 84, 86, 87, 89, 91, 95, 9

### 绘制初始化 A0 热力图

In [6]:
# 五折汇总图
grid_path = OUTPUT_DIR / 'A0_init_all_folds.svg'
plot_A0_heatmap_grid(
    [A0_by_fold[m['tag']] for m in fold_meta],
    [f"Fold {m['fold']}\nseed={m['seed']}" for m in fold_meta],
    save_path=grid_path,
    suptitle=f'Init A0 across {ITRS} folds (topk={args.topk_A0}, subject-mean Pearson)',
)
print(f"\n汇总图已保存: {grid_path}")

# # 保存数值矩阵，便于后续分析
# np.savez(
#     OUTPUT_DIR / 'A0_init_all_folds.npz',
#     **{tag: A0_by_fold[tag] for tag in A0_by_fold},
#     fold_meta=np.array([(m['fold'], m['seed'], m['n_train']) for m in fold_meta]),
#     ch_names=np.array(CH_NAMES),
# )
# print(f"A0 矩阵已保存: {OUTPUT_DIR / 'A0_init_all_folds.npz'}")


汇总图已保存: /root/autodl-tmp/ADSST-Net/figs/Prior-Guided Graph Learning/A0_explainability/A0_init_all_folds.svg


## Prior-Guided Graph Learning (PGL) 的 **U** / **V** 可视化

从训练好的 checkpoint 中读取 `PearsonPriorAdj` 的低秩因子：
- 参数路径：`encoder.encoder_layers.{L}.pglModule.adj.U / .V`
- 形状：`(N, dim_UVA)`，即 `(20 通道, 4)`
- 每折对应 `checkpoints/CAEEG/{project_name}/itr_xxx/checkpoint.pth`

In [10]:
import re
import torch

# ========== checkpoint 路径配置 ==========
PROJECT_NAME = 'CAEEG3_1s_00_EO_Aug-ADSSTNet-NCvsMCIvsAD-oneFreBanDAGnoRanCrop'
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints' / PROJECT_NAME
UV_OUTPUT_DIR = save_path / 'UV'
UV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENCODER_LAYER = 0          # 后续 Alpha / Test 可视化默认层；U/V 部分自动遍历全部层
CMAP_UV = 'RdBu_r'         # U/V 有正负值，用发散色图

fold_dirs = sorted([d for d in CHECKPOINT_ROOT.iterdir() if d.is_dir()]) if CHECKPOINT_ROOT.is_dir() else []
print(f"CHECKPOINT_ROOT = {CHECKPOINT_ROOT}")
print(f"UV_OUTPUT_DIR   = {UV_OUTPUT_DIR}")
print(f"发现折目录 {len(fold_dirs)} 个: {[d.name for d in fold_dirs]}")

CHECKPOINT_ROOT = /root/autodl-tmp/ADSST-Net/checkpoints/CAEEG3_1s_00_EO_Aug-ADSSTNet-NCvsMCIvsAD-oneFreBanDAGnoRanCrop
UV_OUTPUT_DIR   = /root/autodl-tmp/ADSST-Net/figs/Prior-Guided Graph Learning/UV
发现折目录 5 个: ['itr_000', 'itr_001', 'itr_002', 'itr_003', 'itr_004']


In [11]:
from matplotlib.colors import TwoSlopeNorm

UV_CELL_SIZE = 0.30   # 热力图每个方格的边长（inch）


def _normalize_state_key(key: str) -> str:
    return key.replace('module.', '')


def list_gcn_layers(state_dict: dict) -> list[int]:
    """列出 checkpoint 中含 adj.U 的 encoder 层索引。"""
    layers = set()
    pat = re.compile(r'^encoder\.encoder_layers\.(\d+)\.pglModule\.adj\.U$')
    for key in state_dict:
        m = pat.match(_normalize_state_key(key))
        if m:
            layers.add(int(m.group(1)))
    return sorted(layers)


def load_checkpoint_state(fold_dir: Path) -> dict:
    ckpt_path = fold_dir / 'checkpoint.pth'
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"未找到 checkpoint: {ckpt_path}")
    return torch.load(ckpt_path, map_location='cpu')


def extract_UV(state_dict: dict, layer_idx: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """从 state_dict 提取 PearsonPriorAdj 的 U/V，形状均为 (N, r)。"""
    u_key = v_key = None
    u_pat = re.compile(rf'^encoder\.encoder_layers\.{layer_idx}\.pglModule\.adj\.U$')
    v_pat = re.compile(rf'^encoder\.encoder_layers\.{layer_idx}\.pglModule\.adj\.V$')
    for key, tensor in state_dict.items():
        nk = _normalize_state_key(key)
        if u_pat.match(nk):
            u_key = key
        elif v_pat.match(nk):
            v_key = key
    if u_key is None or v_key is None:
        avail = list_gcn_layers(state_dict)
        raise KeyError(
            f"layer {layer_idx} 未找到 adj.U/adj.V；checkpoint 可用层: {avail}"
        )
    U = state_dict[u_key].detach().cpu().numpy()
    V = state_dict[v_key].detach().cpu().numpy()
    return U, V


def _add_uv_grid(ax, n_ch: int, n_rank: int, *, color: str = '#bdbdbd', linewidth: float = 0.5):
    """在 UV 热力图上方绘制方格网格（比 ax.grid 更适合 SVG 导出）。"""
    from matplotlib.collections import LineCollection

    x_edges = np.arange(-0.5, n_rank, 1.0)
    y_edges = np.arange(-0.5, n_ch, 1.0)
    segments = []
    for x in x_edges:
        segments.append([(x, -0.5), (x, n_ch - 0.5)])
    for y in y_edges:
        segments.append([(-0.5, y), (n_rank - 0.5, y)])
    lc = LineCollection(segments, colors=color, linewidths=linewidth, zorder=3)
    ax.add_collection(lc)


def _uv_imshow(ax, mat: np.ndarray, norm, n_ch: int, n_rank: int):
    """在 ax 上绘制方格为正方形的 UV 热力图。"""
    im = ax.imshow(
        mat, cmap=CMAP_UV, norm=norm,
        aspect='equal', interpolation='nearest', origin='upper',
    )
    im.set_rasterized(True)
    im.set_zorder(2)
    _add_uv_grid(ax, n_ch, n_rank)
    ax.set_xlim(-0.5, n_rank - 0.5)
    ax.set_ylim(n_ch - 0.5, -0.5)
    ax.set_xticks(range(n_rank))
    ax.set_yticks(range(n_ch))
    ax.tick_params(axis='both', which='both', length=0, width=0)
    return im


def plot_UV_grid(matrices, fold_labels, param_name: str, save_path: Path, suptitle: str):
    """单层多折对比：1×n_folds 横向排列，共享一个 colorbar。"""
    n_folds = len(matrices)
    n_ch, n_rank = matrices[0].shape
    vlim = float(max(np.max(np.abs(m)) for m in matrices)) or 1e-6
    norm = TwoSlopeNorm(vcenter=0.0, vmin=-vlim, vmax=vlim)

    panel_w = UV_CELL_SIZE * n_rank
    panel_h = UV_CELL_SIZE * n_ch
    gap = 0.40
    cbar_w = 0.22
    cbar_pad = 0.12
    left_margin = 1.05
    bottom_margin = 0.45
    top_margin = 0.55

    fig_w = left_margin + n_folds * panel_w + (n_folds - 1) * gap + cbar_pad + cbar_w + 0.25
    fig_h = bottom_margin + panel_h + top_margin

    fig = plt.figure(figsize=(fig_w, fig_h), facecolor='white')
    bottom_frac = bottom_margin / fig_h
    panel_h_frac = panel_h / fig_h
    panel_w_frac = panel_w / fig_w
    cbar_w_frac = cbar_w / fig_w

    im = None
    for i, (mat, label) in enumerate(zip(matrices, fold_labels)):
        left_frac = (left_margin + i * (panel_w + gap)) / fig_w
        ax = fig.add_axes([left_frac, bottom_frac, panel_w_frac, panel_h_frac])
        im = _uv_imshow(ax, mat, norm, n_ch, n_rank)
        ax.set_title(label, fontsize=FONT_TITLE, fontweight='bold', pad=6)
        ax.set_xticklabels([f'r{j}' for j in range(n_rank)], fontsize=7)
        ax.set_yticklabels(CH_NAMES if i == 0 else [], fontsize=7)

    cbar_left = (left_margin + n_folds * panel_w + (n_folds - 1) * gap + cbar_pad) / fig_w
    cax = fig.add_axes([cbar_left, bottom_frac, cbar_w_frac, panel_h_frac])
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label(f'{param_name} value', fontsize=FONT_AXIS_LABEL)
    cbar.ax.tick_params(labelsize=FONT_TICK)

    fig.suptitle(
        suptitle, fontsize=FONT_TITLE + 1, fontweight='bold',
        y=bottom_frac + panel_h_frac + 0.06,
    )
    save_path = Path(save_path)
    if save_path.suffix.lower() != '.svg':
        save_path = save_path.with_suffix('.svg')
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, format='svg', bbox_inches='tight', facecolor='white')
    plt.close(fig)


def plot_UV_grid_fold_x_layer(
    data_by_fold: dict,
    layer_ids: list[int],
    fold_tags: list[str],
    param_name: str,
    save_path: Path,
    suptitle: str,
):
    """五折汇总：折数×层数网格，行=折、列=层，每行单独 colorbar。"""
    n_folds = len(fold_tags)
    n_layers = len(layer_ids)
    n_ch, n_rank = data_by_fold[fold_tags[0]][layer_ids[0]].shape

    panel_w = UV_CELL_SIZE * n_rank
    panel_h = UV_CELL_SIZE * n_ch
    gap_x = 0.35
    gap_y = 0.32
    cbar_w = 0.18
    cbar_pad = 0.10
    left_margin = 1.05
    bottom_margin = 0.55
    top_margin = 0.70
    right_margin = 0.25

    row_block_w = n_layers * panel_w + (n_layers - 1) * gap_x + cbar_pad + cbar_w
    fig_w = left_margin + row_block_w + right_margin
    fig_h = bottom_margin + n_folds * panel_h + (n_folds - 1) * gap_y + top_margin

    fig = plt.figure(figsize=(fig_w, fig_h), facecolor='white')
    panel_w_frac = panel_w / fig_w
    panel_h_frac = panel_h / fig_h
    cbar_w_frac = cbar_w / fig_w

    for i, fold_tag in enumerate(fold_tags):
        row_mats = [data_by_fold[fold_tag][layer_idx] for layer_idx in layer_ids]
        vlim = float(max(np.max(np.abs(m)) for m in row_mats)) or 1e-6
        norm = TwoSlopeNorm(vcenter=0.0, vmin=-vlim, vmax=vlim)

        bottom_frac = (bottom_margin + (n_folds - 1 - i) * (panel_h + gap_y)) / fig_h
        im = None
        for j, layer_idx in enumerate(layer_ids):
            left_frac = (left_margin + j * (panel_w + gap_x)) / fig_w
            ax = fig.add_axes([left_frac, bottom_frac, panel_w_frac, panel_h_frac])
            im = _uv_imshow(ax, data_by_fold[fold_tag][layer_idx], norm, n_ch, n_rank)
            if i == 0:
                ax.set_title(f'Layer {layer_idx}', fontsize=FONT_TITLE, fontweight='bold', pad=4)
            ax.set_xticklabels([f'r{k}' for k in range(n_rank)], fontsize=7)
            if j == 0:
                ax.set_ylabel(
                    fold_tag, fontsize=FONT_TICK, rotation=0,
                    ha='right', va='center', labelpad=32,
                )
                ax.set_yticklabels(CH_NAMES, fontsize=7)
            else:
                ax.set_yticklabels([], fontsize=7)

        cbar_left = (left_margin + n_layers * panel_w + (n_layers - 1) * gap_x + cbar_pad) / fig_w
        cax = fig.add_axes([cbar_left, bottom_frac, cbar_w_frac, panel_h_frac])
        cbar = fig.colorbar(im, cax=cax)
        if i == 0:
            cbar.set_label(f'{param_name} value', fontsize=FONT_AXIS_LABEL)
        cbar.ax.tick_params(labelsize=7)

    fig.suptitle(
        suptitle, fontsize=FONT_TITLE + 1, fontweight='bold', y=1.0 - top_margin / fig_h / 2,
    )
    plt.savefig(save_path, dpi=600, bbox_inches='tight')
    plt.close(fig)

### 绘制 U/V 热力图

1. **逐层**：每层生成 `U/V_all_folds_layer{L}.png`（1×5 折，共享 colorbar）
2. **五折汇总**：`U/V_all_folds_all_layers.png`（**折数×层数** 网格，每折一行、单独 colorbar）

In [12]:
if not CHECKPOINT_ROOT.is_dir():
    raise FileNotFoundError(f"checkpoint 根目录不存在: {CHECKPOINT_ROOT}")

UV_by_fold = {}   # fold_tag -> {layer_idx: {'U': ..., 'V': ...}}
fold_tags = []
layer_ids = None

for fold_dir in fold_dirs:
    fold_tag = fold_dir.name
    state = load_checkpoint_state(fold_dir)
    avail_layers = list_gcn_layers(state)
    if not avail_layers:
        raise KeyError(f"{fold_tag}: checkpoint 中未找到 pglModule.adj.U/V")

    if layer_ids is None:
        layer_ids = avail_layers
    elif avail_layers != layer_ids:
        raise ValueError(
            f"{fold_tag}: 可用层 {avail_layers} 与其他折不一致（期望 {layer_ids}）"
        )

    UV_by_fold[fold_tag] = {}
    fold_tags.append(fold_tag)
    print(f"{fold_tag}:")

    for layer_idx in layer_ids:
        U, V = extract_UV(state, layer_idx=layer_idx)
        UV_by_fold[fold_tag][layer_idx] = {'U': U, 'V': V}
        print(
            f"  layer {layer_idx}: U{U.shape}, V{V.shape} | "
            f"U [{U.min():.4f}, {U.max():.4f}] | V [{V.min():.4f}, {V.max():.4f}]"
        )

U_by_fold_layer = {
    tag: {layer_idx: UV_by_fold[tag][layer_idx]['U'] for layer_idx in layer_ids}
    for tag in fold_tags
}
V_by_fold_layer = {
    tag: {layer_idx: UV_by_fold[tag][layer_idx]['V'] for layer_idx in layer_ids}
    for tag in fold_tags
}

# 1) 逐层：每层五折横向对比（共享 colorbar）
for layer_idx in layer_ids:
    plot_UV_grid(
        [U_by_fold_layer[tag][layer_idx] for tag in fold_tags],
        fold_tags,
        param_name='U',
        save_path=UV_OUTPUT_DIR / f'U_all_folds_layer{layer_idx}.png',
        suptitle=f'U across folds (encoder layer {layer_idx})',
    )
    plot_UV_grid(
        [V_by_fold_layer[tag][layer_idx] for tag in fold_tags],
        fold_tags,
        param_name='V',
        save_path=UV_OUTPUT_DIR / f'V_all_folds_layer{layer_idx}.png',
        suptitle=f'V across folds (encoder layer {layer_idx})',
    )

# # 2) 五折汇总：折数×层数，每折单独 colorbar
# plot_UV_grid_fold_x_layer(
#     U_by_fold_layer,
#     layer_ids,
#     fold_tags,
#     param_name='U',
#     save_path=UV_OUTPUT_DIR / 'U_all_folds_all_layers.png',
#     suptitle='U across folds and encoder layers',
# )
# plot_UV_grid_fold_x_layer(
#     V_by_fold_layer,
#     layer_ids,
#     fold_tags,
#     param_name='V',
#     save_path=UV_OUTPUT_DIR / 'V_all_folds_all_layers.png',
#     suptitle='V across folds and encoder layers',
# )

npz_payload = {
    'ch_names': np.array(CH_NAMES),
    'layer_ids': np.array(layer_ids),
    'fold_tags': np.array(fold_tags),
}
for tag in fold_tags:
    for layer_idx in layer_ids:
        npz_payload[f'{tag}_layer{layer_idx}_U'] = UV_by_fold[tag][layer_idx]['U']
        npz_payload[f'{tag}_layer{layer_idx}_V'] = UV_by_fold[tag][layer_idx]['V']

np.savez(UV_OUTPUT_DIR / 'UV_all_folds_all_layers.npz', **npz_payload)
print(f"\n逐层图、折×层汇总图与数值已保存: {UV_OUTPUT_DIR}")
print(f"共 {len(layer_ids)} 层 × {len(fold_tags)} 折")

itr_000:
  layer 0: U(20, 4), V(20, 4) | U [-0.0644, 0.0568] | V [-0.0546, 0.0622]
  layer 1: U(20, 4), V(20, 4) | U [-0.0409, 0.0489] | V [-0.0513, 0.0401]
  layer 2: U(20, 4), V(20, 4) | U [-0.0570, 0.0713] | V [-0.0540, 0.0765]
  layer 3: U(20, 4), V(20, 4) | U [-0.0796, 0.0522] | V [-0.0842, 0.0496]
  layer 4: U(20, 4), V(20, 4) | U [-0.0424, 0.0627] | V [-0.0585, 0.0551]
  layer 5: U(20, 4), V(20, 4) | U [-0.0482, 0.0288] | V [-0.0443, 0.0331]
itr_001:
  layer 0: U(20, 4), V(20, 4) | U [-0.2287, 0.2292] | V [-0.2202, 0.2432]
  layer 1: U(20, 4), V(20, 4) | U [-0.1651, 0.1537] | V [-0.1681, 0.1349]
  layer 2: U(20, 4), V(20, 4) | U [-0.2653, 0.2068] | V [-0.2513, 0.2187]
  layer 3: U(20, 4), V(20, 4) | U [-0.2382, 0.1102] | V [-0.2163, 0.1191]
  layer 4: U(20, 4), V(20, 4) | U [-0.1532, 0.1347] | V [-0.1549, 0.1367]
  layer 5: U(20, 4), V(20, 4) | U [-0.1577, 0.1468] | V [-0.1569, 0.1700]
itr_002:
  layer 0: U(20, 4), V(20, 4) | U [-0.0447, 0.0552] | V [-0.0383, 0.0489]
  layer 1: 

## 测试集 adjacency_matrix / A_learn 可视化

-**注：** 这部分需要在服务器上运行

对每一折 checkpoint：
1. 按 `exp_supervised._build_model` 初始化 ADSSTNet 并加载 `checkpoint.pth`（SWA）
2. 加载 **TEST** 集，前向推理收集**全部 encoder 层**的 `adjacency_matrix` 与 `A_learn`
3. 每层按 NC / MCI / AD 分组求均值，绘制三类对比热力图
4. 额外生成 **折数×层数** 汇总图（每折单独 colorbar）

In [19]:
import torch
from torch.optim.swa_utils import AveragedModel
from torch.utils.data import DataLoader

from data_provider.data_factory import data_provider
from data_provider.uea import collate_fn
from models import ADSSTNet

# ========== 测试集 A_learn / adjacency 可视化配置 ==========
TEST_ADJ_OUTPUT_DIR = save_path / 'Adj_matrix_A_learn'
TEST_ADJ_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ADJ_LAYER = 0                 # Alpha 可视化默认层；Test 部分自动遍历全部 encoder 层
USE_SWA = True                # 与训练脚本 --swa 一致
CLASS_NAMES = ['NC', 'MCI', 'AD']  # remapped label 0/1/2
ADJ_CELL_SIZE = 0.22            # 折×层汇总图每个方格边长（inch）

MODEL_ARGS_BASE = dict(
    task_name='supervised',
    model='ADSSTNet',
    data='CAEEG',
    root_path=str((PROJECT_ROOT / 'dataset/CAEEG3_1.0s_0.0_EO_Aug').resolve()) + '/',
    selected_classes=[0, 1, 2],
    cross_val='mccv',
    use_augmentation=True,
    augmentations='FreBandMix',
    e_layers=0,
    d_model=20,
    expand=2,
    d_state=16,
    dt_rank=2,
    d_conv_in=4,
    d_conv=4,
    d_conv_out=4,
    bimamba=True,
    space=True,
    se_reduction=16,
    dim_UVA=4,
    topk_A0=4,
    embedding_dim=32,
    sample_adapt_A0=True,
    sample_adapt_Fre=True,
    projection_dim=128,
    dropout_head=0.7,
    num_heads=4,
    downsample_min_len=4,
    group_size=16,
    batch_size=256,
    num_workers=0,
    preserve_frontal_occipital=False,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"TEST_ADJ_OUTPUT_DIR = {TEST_ADJ_OUTPUT_DIR}")
print(f"DEVICE = {DEVICE}, USE_SWA = {USE_SWA}")

TEST_ADJ_OUTPUT_DIR = /root/autodl-tmp/ADSST-Net/figs/Prior-Guided Graph Learning/Adj_matrix_A_learn
DEVICE = cuda, USE_SWA = True


In [20]:
def _get_data(flag: str, fold_seed: int):
    """复现 exp_supervised._get_data：固定 seed 后加载数据集。"""
    fold_args = Namespace(**MODEL_ARGS_BASE, seed=fold_seed)
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    return data_provider(fold_args, flag, log=None)


def build_adsstnet_for_fold(fold_seed: int):
    """复现 exp_supervised._build_model：按 TRAIN 集确定维度并计算 A0。"""
    fold_args = Namespace(**MODEL_ARGS_BASE, seed=fold_seed)
    random.seed(fold_seed)
    np.random.seed(fold_seed)

    train_data, _ = data_provider(fold_args, 'TRAIN', log=None)
    fold_args.seq_len = train_data.max_seq_len
    fold_args.enc_in = train_data.X.shape[-1]
    # 与 checkpoint 的分类头一致。某一折的训练集可能缺一个类别，不能用 unique(y) 推断。
    if fold_args.selected_classes:
        fold_args.num_class = len(fold_args.selected_classes)
    else:
        fold_args.num_class = len(np.unique(train_data.y[:, 0]))
    fold_args.A0 = compute_pearson_A0_subject_mean(
        train_data, enc_in=fold_args.enc_in, **A0_KWARGS,
    )

    model = ADSSTNet.Model(fold_args).float().to(DEVICE)
    if USE_SWA:
        model = AveragedModel(model)
    model.eval()
    return model, fold_args


def load_test_loader(fold_args):
    """加载 TEST 集 DataLoader（与 exp_supervised.test 一致）。"""
    test_data, _ = _get_data('TEST', fold_args.seed)
    test_loader = DataLoader(
        test_data,
        batch_size=fold_args.batch_size,
        shuffle=False,
        num_workers=fold_args.num_workers,
        drop_last=False,
        collate_fn=lambda x: collate_fn(x, max_len=fold_args.seq_len),
    )
    return test_data, test_loader


def load_fold_checkpoint(model, fold_dir: Path):
    state = torch.load(fold_dir / 'checkpoint.pth', map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()


def _as_layer_list(x):
    if isinstance(x, (list, tuple)):
        return list(x)
    return [x]


COLLECT_AUX_KEYS = ('A_learn', 'delta', 'gate')


def _aux_layers_by_key(aux, keys=COLLECT_AUX_KEYS):
    """从 aux_list 按 keys 提取每层 tensor（缺失或非 dict 时为 None）。"""
    out = {k: [] for k in keys}
    for layer_aux in _as_layer_list(aux):
        if isinstance(layer_aux, dict):
            for k in keys:
                out[k].append(layer_aux.get(k))
        else:
            if 'A_learn' in out:
                out['A_learn'].append(layer_aux)
            for k in keys:
                if k != 'A_learn':
                    out[k].append(None)
    return out


def _append_batch_samples(target_by_layer, layer_idx, tensor, sample_indices):
    """按 sample_indices 将 batch 中对应样本追加到 target_by_layer[layer_idx]。"""
    n = len(sample_indices)
    if tensor is None:
        target_by_layer[layer_idx].extend([None] * n)
        return
    arr = tensor.detach().cpu().numpy()
    target_by_layer[layer_idx].extend(arr[i] for i in sample_indices)


def _balance_class_indices(
    labels,
    class_ids,
    rng: np.random.Generator,
) -> tuple[list[int], int, dict[int, int]]:
    """按各类别最少样本数随机下采样，返回保留的全局样本索引。"""
    labels = np.asarray(labels)
    class_ids = sorted(class_ids)
    per_class_idx = {c: np.where(labels == c)[0] for c in class_ids}
    counts = {c: len(per_class_idx[c]) for c in class_ids}
    missing = [c for c in class_ids if counts[c] == 0]
    if missing:
        raise RuntimeError(f'类别 {missing} 无样本，无法均衡采样')

    min_n = min(counts.values())
    selected = []
    for c in class_ids:
        idx = per_class_idx[c]
        if len(idx) == min_n:
            chosen = idx
        else:
            chosen = rng.choice(idx, size=min_n, replace=False)
        selected.extend(int(i) for i in chosen)
    rng.shuffle(selected)
    return selected, min_n, counts


def _subset_collected_data(data_by_key, label_list, keep_indices):
    """按 keep_indices 同步裁剪各 key / 各层的样本列表。"""
    label_list = [label_list[i] for i in keep_indices]
    subset = {}
    for key, by_layer in data_by_key.items():
        subset[key] = {
            layer_idx: [by_layer[layer_idx][i] for i in keep_indices]
            for layer_idx in by_layer
        }
    return subset, label_list


def _pgl_layers_from_vis(pgl_vis_list, aux_keys=COLLECT_AUX_KEYS):
    """从 PGL_visualization_list 提取 adjacency_matrix 与各 aux 字段。"""
    adj_layers = []
    aux_list = []
    for pgl_dict in _as_layer_list(pgl_vis_list):
        if not isinstance(pgl_dict, dict):
            raise TypeError(
                f'PGL_visualization 应为 dict，实际为 {type(pgl_dict).__name__}'
            )
        if 'adjacency_matrix' not in pgl_dict:
            raise KeyError(
                f'PGL_visualization 缺少键 adjacency_matrix，'
                f'现有键: {list(pgl_dict.keys())}'
            )
        adj_layers.append(pgl_dict['adjacency_matrix'])
        aux_list.append(pgl_dict.get('aux'))
    aux_by_key = _aux_layers_by_key(aux_list, keys=aux_keys)
    return adj_layers, aux_by_key


@torch.no_grad()
def collect_test_adjacency_all_layers(
    model,
    test_loader,
    aux_keys=COLLECT_AUX_KEYS,
    correct_only=True,
    balance_classes=True,
    balance_seed=None,
    class_ids=None,
):
    """在测试集上收集全部 encoder 层的 PGL_visualization（adjacency 与 aux 字段）。

    Parameters
    ----------
    correct_only : bool
        为 True 时仅保留分类正确的样本（argmax(logits) == 真实标签）。
    balance_classes : bool
        为 True 时按各类别最少样本数随机下采样，使 NC / MCI / AD 样本量一致。
    balance_seed : int, optional
        类别均衡采样的随机种子；为 None 时使用全局 numpy 随机状态。
    class_ids : sequence, optional
        参与均衡的类别 id；默认取 ``label_list`` 中出现过的全部类别。

    Returns
    -------
    data_by_key : dict[str, dict[int, list]]
        例如 ``data_by_key['adjacency'][layer_idx]``、``data_by_key['A_learn'][layer_idx]``、
        ``data_by_key['delta'][layer_idx]``、``data_by_key['gate'][layer_idx]``；
        每个 list 与 ``label_list`` 等长，元素为单样本 ``np.ndarray`` 或 ``None``。
    label_list, layer_ids
    """
    data_by_key = None
    label_list = []
    aux_keys = tuple(aux_keys)
    n_total = n_kept = 0

    for batch_x, label_id, _ in test_loader:
        batch_x_ori = batch_x[:, 0, :, :].float().to(DEVICE)
        labels = label_id[:, 0].cpu().numpy().astype(int)
        n_samples = len(labels)
        n_total += n_samples

        _, logits, _, _, pgl_vis_list, _ = model(
            batch_x_ori, None, None, None, model_state='test',
        )
        preds = logits.argmax(dim=-1).detach().cpu().numpy().astype(int)
        if correct_only:
            sample_indices = [i for i in range(n_samples) if preds[i] == labels[i]]
        else:
            sample_indices = list(range(n_samples))
        if not sample_indices:
            continue
        n_kept += len(sample_indices)

        adj_layers, aux_by_key = _pgl_layers_from_vis(pgl_vis_list, aux_keys=aux_keys)
        n_layers = len(adj_layers)

        for k, layer_tensors in aux_by_key.items():
            if len(layer_tensors) != n_layers:
                raise ValueError(
                    f"aux['{k}'] 层数 {len(layer_tensors)} 与 adjacency 层数 {n_layers} 不一致"
                )

        if data_by_key is None:
            data_by_key = {'adjacency': {layer_idx: [] for layer_idx in range(n_layers)}}
            for k in aux_keys:
                data_by_key[k] = {layer_idx: [] for layer_idx in range(n_layers)}

        for layer_idx, adj_tensor in enumerate(adj_layers):
            _append_batch_samples(data_by_key['adjacency'], layer_idx, adj_tensor, sample_indices)
            for k in aux_keys:
                _append_batch_samples(
                    data_by_key[k], layer_idx, aux_by_key[k][layer_idx], sample_indices,
                )

        label_list.extend(int(labels[i]) for i in sample_indices)

    if data_by_key is None:
        raise RuntimeError('未收集到任何样本（测试集为空或 correct_only 下无正确分类样本）')
    if correct_only:
        print(f'collect_test_adjacency: 保留正确分类样本 {n_kept}/{n_total}')

    if balance_classes:
        balance_class_ids = sorted(set(class_ids if class_ids is not None else label_list))
        rng = np.random.default_rng(balance_seed)
        keep_indices, min_n, raw_counts = _balance_class_indices(
            label_list, balance_class_ids, rng,
        )
        data_by_key, label_list = _subset_collected_data(data_by_key, label_list, keep_indices)
        count_str = ', '.join(
            f"{CLASS_NAMES[c] if c < len(CLASS_NAMES) else c}: {raw_counts[c]}->{min_n}"
            for c in balance_class_ids
        )
        print(f'collect_test_adjacency: 类别均衡采样 -> 每类 n={min_n} ({count_str})')

    layer_ids = sorted(data_by_key['adjacency'].keys())
    return data_by_key, label_list, layer_ids


def mean_matrices_by_class(mats, labels, class_ids):
    """按类别求矩阵均值。"""
    means = {}
    labels = np.asarray(labels)
    for c in class_ids:
        idx = np.where(labels == c)[0]
        if len(idx) == 0:
            continue
        means[c] = np.mean([mats[i] for i in idx], axis=0)
    return means


def subtract_A0_from_sample_mats(mats, A0: np.ndarray):
    """逐样本邻接矩阵减去该折初始化 A0。"""
    A0 = np.asarray(A0, dtype=np.float64)
    out = []
    for m in mats:
        if m is None:
            out.append(None)
        else:
            out.append(np.asarray(m, dtype=np.float64) - A0)
    return out


def subtract_A0_from_matrix(mat: np.ndarray, A0: np.ndarray) -> np.ndarray:
    return np.asarray(mat, dtype=np.float64) - np.asarray(A0, dtype=np.float64)


MINUS_A0_PLOT_KW = dict(
    cmap_name=CMAP_ADJ_MINUS_A0,
    diverging=True,
    value_label='adj − A0',
)


def plot_class_mean_grid(
    class_means,
    counts,
    class_names,
    save_path: Path,
    suptitle: str,
    *,
    cmap_name: str | None = None,
    diverging: bool = False,
    value_label: str = 'Edge weight',
):
    """NC / MCI / AD 三类均值矩阵汇总热力图。"""
    class_ids = sorted(class_means.keys())
    mats, panel_labels = [], []
    for c in class_ids:
        mats.append(class_means[c].copy())
        panel_labels.append(f"{class_names[c]}\n(n={counts[c]})")
    plot_A0_heatmap_grid(
        mats, panel_labels, save_path, suptitle,
        cmap_name=cmap_name, diverging=diverging, value_label=value_label,
    )


def _adj_imshow(
    ax,
    mat: np.ndarray,
    vmin: float,
    vmax: float,
    n_ch: int,
    *,
    cmap_name: str | None = None,
    diverging: bool = False,
):
    """在 ax 上绘制方格为正方形的 adjacency / A_learn 热力图。"""
    cmap = _cmap_for_adj_plot(cmap_name)
    m = _prep_adj_matrix(mat, diverging=diverging)
    im = _imshow_adj_matrix(ax, m, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xlim(-0.5, n_ch - 0.5)
    ax.set_ylim(n_ch - 0.5, -0.5)
    ax.set_xticks(range(n_ch))
    ax.set_yticks(range(n_ch))
    ax.set_xticks(np.arange(-0.5, n_ch, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_ch, 1), minor=True)
    ax.grid(which='minor', color='#d8d8d8', linestyle='-', linewidth=0.35, alpha=0.9)
    ax.set_axisbelow(True)
    ax.tick_params(which='minor', bottom=False, left=False, length=0)
    return im


def plot_adj_fold_x_layer_grid(
    data_by_fold: dict,
    layer_ids: list[int],
    fold_tags: list[str],
    save_path: Path,
    suptitle: str,
    value_label: str = 'Edge weight',
    *,
    cmap_name: str | None = None,
    diverging: bool = False,
):
    """折数×层数汇总：行=折、列=层，每行单独 colorbar。"""
    n_folds = len(fold_tags)
    n_layers = len(layer_ids)
    n_ch = len(CH_NAMES)

    panel_w = ADJ_CELL_SIZE * n_ch
    panel_h = ADJ_CELL_SIZE * n_ch
    gap_x = 0.35
    gap_y = 0.32
    cbar_w = 0.18
    cbar_pad = 0.10
    left_margin = 1.05
    bottom_margin = 0.55
    top_margin = 0.70
    right_margin = 0.25

    fig_w = left_margin + n_layers * panel_w + (n_layers - 1) * gap_x + cbar_pad + cbar_w + right_margin
    fig_h = bottom_margin + n_folds * panel_h + (n_folds - 1) * gap_y + top_margin

    fig = plt.figure(figsize=(fig_w, fig_h), facecolor='white')
    panel_w_frac = panel_w / fig_w
    panel_h_frac = panel_h / fig_h
    cbar_w_frac = cbar_w / fig_w

    for i, fold_tag in enumerate(fold_tags):
        row_mats = [data_by_fold[fold_tag][layer_idx] for layer_idx in layer_ids]
        row_mats_plot = [_prep_adj_matrix(m, diverging=diverging) for m in row_mats]
        vmin, vmax = _vmin_vmax_from_mats(row_mats_plot, diverging=diverging)

        bottom_frac = (bottom_margin + (n_folds - 1 - i) * (panel_h + gap_y)) / fig_h
        im = None
        for j, layer_idx in enumerate(layer_ids):
            left_frac = (left_margin + j * (panel_w + gap_x)) / fig_w
            ax = fig.add_axes([left_frac, bottom_frac, panel_w_frac, panel_h_frac])
            im = _adj_imshow(
                ax, row_mats[j], vmin, vmax, n_ch,
                cmap_name=cmap_name, diverging=diverging,
            )
            if i == 0:
                ax.set_title(f'Layer {layer_idx}', fontsize=FONT_TITLE, fontweight='bold', pad=4)
            if j == 0:
                ax.set_ylabel(
                    fold_tag, fontsize=FONT_TICK, rotation=0,
                    ha='right', va='center', labelpad=32,
                )
                ax.set_yticklabels(CH_NAMES, fontsize=6)
            else:
                ax.set_yticklabels([], fontsize=6)
            ax.set_xticklabels(CH_NAMES if i == n_folds - 1 else [], rotation=90, ha='center', fontsize=6)

        cbar_left = (left_margin + n_layers * panel_w + (n_layers - 1) * gap_x + cbar_pad) / fig_w
        cax = fig.add_axes([cbar_left, bottom_frac, cbar_w_frac, panel_h_frac])
        cbar = fig.colorbar(im, cax=cax)
        if i == 0:
            cbar.set_label(value_label, fontsize=FONT_AXIS_LABEL)
        cbar.ax.tick_params(labelsize=7)

    fig.suptitle(
        suptitle, fontsize=FONT_TITLE + 1, fontweight='bold', y=1.0 - top_margin / fig_h / 2,
    )
    plt.savefig(save_path, dpi=600, bbox_inches='tight')
    plt.close(fig)


import networkx as nx
from matplotlib import colors as mcolors
from nilearn import plotting

# EEG 通道 MNI 坐标（与 CH_NAMES 顺序一致）
EEG_CHANNEL_COORDS = np.array([
    [-19.,  59.,   -2.],    # Fp1
    [ 19.,  59.,   -2.],    # Fp2
    [  0.,  18.,   48.],    # Fz
    [-21.,  22.,   24.],    # F3
    [ 21.,  22.,   24.],    # F4
    [-42.,  30.,   -2.],    # F7
    [ 42.,  30.,   -2.],    # F8
    [  0.,  -20.,  70.],    # Cz
    [-30.,  -20.,  48.],    # C3
    [ 30.,  -20.,  48.],    # C4
    [-60.,  -20.,  -2.],    # T7
    [ 60.,  -20.,  -2.],    # T8
    [  0.,  -50.,  48.],    # Pz
    [-21.,  -54.,  24.],    # P3
    [ 21.,  -54.,  24.],    # P4
    [-42.,  -62.,  -2.],    # P7
    [ 42.,  -62.,  -2.],    # P8
    [  0.,  -92.,  -2.],    # Oz
    [-19.,  -88.,  -2.],    # O1
    [ 19.,  -88.,  -2.],    # O2
])


def symmetrize_A0_for_plot(A: np.ndarray) -> np.ndarray:
    """矩阵一般不对称，connectome 绘图前先对称化。"""
    A = np.asarray(A, dtype=np.float64)
    A_plot = 0.5 * (A + A.T)
    np.fill_diagonal(A_plot, 0.0)
    return A_plot


def _prep_connectome_matrix(mat: np.ndarray, *, symmetrize: bool = False) -> np.ndarray:
    m = np.asarray(mat, dtype=np.float64).copy()
    if symmetrize:
        m = symmetrize_A0_for_plot(m)
    else:
        np.fill_diagonal(m, 0.0)
    return m


def _connectome_node_sizes(mat: np.ndarray, n_nodes: int) -> np.ndarray:
    G = nx.from_numpy_array(np.abs(mat))
    degrees = dict(nx.degree(G))
    return np.array([degrees.get(i, 0) + 30 for i in range(n_nodes)])


def _vmin_vmax_from_connectome_mats(mats, *, diverging: bool = False, symmetrize: bool = False) -> tuple[float, float]:
    parts = []
    for mat in mats:
        m = _prep_connectome_matrix(mat, symmetrize=symmetrize)
        off = m[~np.eye(m.shape[0], dtype=bool)]
        parts.append(off[np.isfinite(off)])
    vals = np.concatenate(parts) if parts else np.array([])
    if vals.size == 0:
        return (-1.0, 1.0) if diverging else (0.0, 1.0)
    vmin = float(np.min(vals))
    vmax = float(np.max(vals))
    if diverging:
        vabs = max(abs(vmin), abs(vmax), 1e-6)
        return -vabs, vabs
    if vmin >= vmax:
        vmax = vmin + 1e-6
    return vmin, vmax


def _plot_connectome_on_ax(
    ax,
    mat: np.ndarray,
    *,
    coords: np.ndarray,
    vmin: float,
    vmax: float,
    edge_cmap: str,
    linewidth: float = 1.5,
    node_sizes=None,
    n_nodes: int,
    symmetrize: bool = False,
):
    mat_plot = _prep_connectome_matrix(mat, symmetrize=symmetrize)
    if node_sizes is None:
        node_sizes = _connectome_node_sizes(mat_plot, n_nodes)
    return plotting.plot_connectome(
        adjacency_matrix=mat_plot,
        node_coords=coords,
        node_color='auto',
        node_size=node_sizes,
        edge_cmap=edge_cmap,
        edge_threshold=0,
        edge_kwargs={'linewidth': linewidth},
        edge_vmin=vmin,
        edge_vmax=vmax,
        display_mode='z',
        figure=ax.figure,
        axes=ax,
        colorbar=False,
    )


def plot_connectome_grid(
    mats,
    panel_labels,
    save_path: Path,
    suptitle: str,
    *,
    cmap_name: str | None = None,
    diverging: bool = False,
    value_label: str = 'Edge weight',
    linewidth: float = 1.5,
    dpi: int = 300,
    coords=None,
    symmetrize: bool = False,
    shared_colorbar: bool = True,
):
    """多面板顶视 connectome 汇总图。"""
    n_panels = len(mats)
    n_nodes = len(CH_NAMES)
    coords = np.asarray(coords if coords is not None else EEG_CHANNEL_COORDS, dtype=np.float64)

    mats_plot = [_prep_connectome_matrix(m, symmetrize=symmetrize) for m in mats]
    if cmap_name is not None:
        edge_cmap = cmap_name
    elif diverging:
        edge_cmap = CMAP_ADJ_MINUS_A0
    else:
        edge_cmap = CMAP_A0_OPTIONS.get(CMAP_A0_NAME, 'YlOrRd')

    displays = []
    if shared_colorbar:
        vmin, vmax = _vmin_vmax_from_connectome_mats(mats_plot, diverging=diverging, symmetrize=False)
        fig = plt.figure(figsize=(4.8 * n_panels + 0.9, 5.2), facecolor='white')
        gs = fig.add_gridspec(1, n_panels + 1, width_ratios=[1] * n_panels + [0.04], wspace=0.12)

        for i, (mat, label) in enumerate(zip(mats, panel_labels)):
            ax = fig.add_subplot(gs[0, i])
            ax.set_title(label, fontsize=FONT_TITLE, fontweight='bold')
            disp = _plot_connectome_on_ax(
                ax, mat, coords=coords, vmin=vmin, vmax=vmax,
                edge_cmap=edge_cmap, linewidth=linewidth, n_nodes=n_nodes,
                symmetrize=symmetrize,
            )
            displays.append(disp)

        cax = fig.add_subplot(gs[0, n_panels])
        _set_subplot_white_bg(cax)
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        sm = plt.cm.ScalarMappable(cmap=plt.colormaps[edge_cmap], norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, cax=cax)
        cbar.set_label(value_label, fontsize=FONT_AXIS_LABEL)
        cbar.ax.tick_params(labelsize=FONT_TICK)
    else:
        fig = plt.figure(figsize=(6.2 * n_panels, 5.4), facecolor='white')
        outer_gs = fig.add_gridspec(1, n_panels, wspace=0.2)

        for i, (mat, label) in enumerate(zip(mats, panel_labels)):
            panel_gs = outer_gs[i].subgridspec(1, 2, width_ratios=[1, 0.07], wspace=0.06)
            ax = fig.add_subplot(panel_gs[0, 0])
            ax.set_title(label, fontsize=FONT_TITLE, fontweight='bold')
            vmin_i, vmax_i = _vmin_vmax_from_connectome_mats(
                [mats_plot[i]], diverging=diverging, symmetrize=False,
            )
            disp = _plot_connectome_on_ax(
                ax, mat, coords=coords, vmin=vmin_i, vmax=vmax_i,
                edge_cmap=edge_cmap, linewidth=linewidth, n_nodes=n_nodes,
                symmetrize=symmetrize,
            )
            displays.append(disp)

            cax = fig.add_subplot(panel_gs[0, 1])
            _set_subplot_white_bg(cax)
            norm = mcolors.Normalize(vmin=vmin_i, vmax=vmax_i)
            sm = plt.cm.ScalarMappable(cmap=plt.colormaps[edge_cmap], norm=norm)
            sm.set_array([])
            cbar = fig.colorbar(sm, cax=cax)
            cbar.set_label(value_label, fontsize=FONT_AXIS_LABEL)
            cbar.ax.tick_params(labelsize=FONT_TICK)

    fig.suptitle(suptitle, fontsize=FONT_TITLE + 1, fontweight='bold', y=1.02)
    save_filename = Path(save_path)
    if save_filename.suffix.lower() != '.svg':
        save_filename = save_filename.with_suffix('.svg')
    save_filename.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_filename, format='svg', bbox_inches='tight', facecolor='white')
    for disp in displays:
        disp.close()
    plt.close(fig)


def plot_class_mean_connectome_grid(
    class_means,
    counts,
    class_names,
    save_path: Path,
    suptitle: str,
    *,
    cmap_name: str | None = None,
    diverging: bool = False,
    value_label: str = 'Edge weight',
    linewidth: float = 1.5,
    dpi: int = 300,
    symmetrize: bool = True,
):
    """NC / MCI / AD 三类均值顶视 connectome 汇总图（每子图独立 colorbar）。"""
    class_ids = sorted(class_means.keys())
    mats = [class_means[c] for c in class_ids]
    panel_labels = [f"{class_names[c]}\n(n={counts[c]})" for c in class_ids]
    plot_connectome_grid(
        mats, panel_labels, save_path, suptitle,
        cmap_name=cmap_name, diverging=diverging, value_label=value_label,
        linewidth=linewidth, dpi=dpi, symmetrize=symmetrize,
        shared_colorbar=False,
    )

### 绘制 adjacency_matrix / A_learn 热力图

In [21]:
if not CHECKPOINT_ROOT.is_dir():
    raise FileNotFoundError(f"checkpoint 根目录不存在: {CHECKPOINT_ROOT}")

test_stats_by_fold = {}
fold_tags = []
layer_ids = None

for fold_idx, fold_dir in enumerate(fold_dirs):
    fold_seed = BASE_SEED + fold_idx + 1
    fold_tag = fold_dir.name

    print(f"\n{'=' * 60}")
    print(f"{fold_tag} | seed={fold_seed}")
    print('=' * 60)

    model, fold_args = build_adsstnet_for_fold(fold_seed)
    load_fold_checkpoint(model, fold_dir)
    _, test_loader = load_test_loader(fold_args)

    data_by_key, label_list, fold_layer_ids = collect_test_adjacency_all_layers(
        model, test_loader, aux_keys=('A_learn',)
    )
    adj_by_layer = data_by_key['adjacency']
    alearn_by_layer = data_by_key['A_learn']
    if layer_ids is None:
        layer_ids = fold_layer_ids
    elif fold_layer_ids != layer_ids:
        raise ValueError(
            f"{fold_tag}: encoder 层 {fold_layer_ids} 与其他折不一致（期望 {layer_ids}）"
        )

    labels_np = np.asarray(label_list)
    class_ids = sorted(set(label_list))
    counts = {c: int(np.sum(labels_np == c)) for c in class_ids}

    if fold_tag not in A0_by_fold:
        raise KeyError(f'{fold_tag} 不在 A0_by_fold，请先运行 A0 初始化 cell')
    A0_fold = np.asarray(A0_by_fold[fold_tag], dtype=np.float64)

    fold_stats = {}
    npz_payload = {
        'ch_names': np.array(CH_NAMES),
        'layer_ids': np.array(layer_ids),
        'fold_tag': fold_tag,
        'fold_seed': fold_seed,
        'A0': A0_fold,
        'adj': adj_by_layer,
        'label_list': np.array(label_list, dtype=int),
        'alearn': alearn_by_layer
    }

    for layer_idx in layer_ids:
        adj_means = mean_matrices_by_class(adj_by_layer[layer_idx], label_list, class_ids)
        adj_residual_mats = subtract_A0_from_sample_mats(adj_by_layer[layer_idx], A0_fold)
        adj_means_minus_A0 = mean_matrices_by_class(adj_residual_mats, label_list, class_ids)
        alearn_means = mean_matrices_by_class(alearn_by_layer[layer_idx], label_list, class_ids)
        adj_overall = np.mean(adj_by_layer[layer_idx], axis=0)
        adj_overall_minus_A0 = subtract_A0_from_matrix(adj_overall, A0_fold)
        alearn_overall = np.mean(alearn_by_layer[layer_idx], axis=0)

        fold_stats[layer_idx] = {
            'adj_means': adj_means,
            'adj_means_minus_A0': adj_means_minus_A0,
            'alearn_means': alearn_means,
            'adj_overall': adj_overall,
            'adj_overall_minus_A0': adj_overall_minus_A0,
            'alearn_overall': alearn_overall,
        }

        for c in class_ids:
            print(f"  layer {layer_idx} | {CLASS_NAMES[c]}: n={counts[c]}")

        # plot_class_mean_connectome_grid(
        #     adj_means_minus_A0, counts, CLASS_NAMES,
        #     save_path=TEST_ADJ_OUTPUT_DIR / f'adjacency_mean_by_class_{fold_tag}_layer{layer_idx}_minus_A0_connectome',
        #     suptitle=f'Test adjacency (mean − A0) | {fold_tag} | layer={layer_idx}',
        #     linewidth=1.5,
        #     dpi=600,
        #     **MINUS_A0_PLOT_KW,
        # )

        plot_class_mean_grid(
            adj_means, counts, CLASS_NAMES,
            save_path=TEST_ADJ_OUTPUT_DIR / f'adjacency_mean_by_class_{fold_tag}_layer{layer_idx}.png',
            suptitle=f'Test adjacency (mean) | {fold_tag} | layer={layer_idx}',
        )

        # plot_class_mean_grid(
        #     adj_means_minus_A0, counts, CLASS_NAMES,
        #     save_path=TEST_ADJ_OUTPUT_DIR / f'adjacency_mean_by_class_{fold_tag}_layer{layer_idx}_minus_A0.png',
        #     suptitle=f'Test adjacency (mean − A0) | {fold_tag} | layer={layer_idx}',
        #     **MINUS_A0_PLOT_KW,
        # )

        plot_class_mean_grid(
            alearn_means, counts, CLASS_NAMES,
            save_path=TEST_ADJ_OUTPUT_DIR / f'A_learn_mean_by_class_{fold_tag}_layer{layer_idx}.png',
            suptitle=f'Test A_learn (mean) | {fold_tag} | layer={layer_idx}',
        )

        for c in class_ids:
            npz_payload[f'adj_mean_{CLASS_NAMES[c]}_layer{layer_idx}'] = adj_means[c]
            npz_payload[f'adj_mean_{CLASS_NAMES[c]}_layer{layer_idx}_minus_A0'] = adj_means_minus_A0[c]
            npz_payload[f'A_learn_mean_{CLASS_NAMES[c]}_layer{layer_idx}'] = alearn_means[c]
            npz_payload[f'count_{CLASS_NAMES[c]}'] = counts[c]
        npz_payload[f'adj_overall_mean_layer{layer_idx}'] = adj_overall
        npz_payload[f'adj_overall_mean_layer{layer_idx}_minus_A0'] = adj_overall_minus_A0
        npz_payload[f'A_learn_overall_mean_layer{layer_idx}'] = alearn_overall

    np.savez(TEST_ADJ_OUTPUT_DIR / f'test_adj_stats_{fold_tag}.npz', **npz_payload)
    test_stats_by_fold[fold_tag] = fold_stats
    fold_tags.append(fold_tag)
    print(f"  已保存至 {TEST_ADJ_OUTPUT_DIR}")

# adj_overall_by_fold_layer = {
#     tag: {layer_idx: test_stats_by_fold[tag][layer_idx]['adj_overall'] for layer_idx in layer_ids}
#     for tag in fold_tags
# }
# adj_overall_minus_A0_by_fold_layer = {
#     tag: {layer_idx: test_stats_by_fold[tag][layer_idx]['adj_overall_minus_A0'] for layer_idx in layer_ids}
#     for tag in fold_tags
# }
# alearn_overall_by_fold_layer = {
#     tag: {layer_idx: test_stats_by_fold[tag][layer_idx]['alearn_overall'] for layer_idx in layer_ids}
#     for tag in fold_tags
# }

# # 逐层：每层五折横向对比（测试集全体样本均值）
# for layer_idx in layer_ids:
#     plot_A0_heatmap_grid(
#         [adj_overall_by_fold_layer[tag][layer_idx] for tag in fold_tags],
#         fold_tags,
#         save_path=TEST_ADJ_OUTPUT_DIR / f'adjacency_overall_mean_all_folds_layer{layer_idx}.png',
#         suptitle=f'Test adjacency overall mean | layer={layer_idx}',
#     )
#     plot_A0_heatmap_grid(
#         [adj_overall_minus_A0_by_fold_layer[tag][layer_idx] for tag in fold_tags],
#         fold_tags,
#         save_path=TEST_ADJ_OUTPUT_DIR / f'adjacency_overall_mean_all_folds_layer{layer_idx}_minus_A0.png',
#         suptitle=f'Test adjacency overall mean − A0 | layer={layer_idx}',
#         **MINUS_A0_PLOT_KW,
#     )
#     plot_A0_heatmap_grid(
#         [alearn_overall_by_fold_layer[tag][layer_idx] for tag in fold_tags],
#         fold_tags,
#         save_path=TEST_ADJ_OUTPUT_DIR / f'A_learn_overall_mean_all_folds_layer{layer_idx}.png',
#         suptitle=f'Test A_learn overall mean | layer={layer_idx}',
#     )

# # 按类别：折×层汇总（NC / MCI / AD 各一张）
# for c in sorted({c for tag in fold_tags for c in test_stats_by_fold[tag][layer_ids[0]]['adj_means']}):
#     adj_class_by_fold_layer = {
#         tag: {layer_idx: test_stats_by_fold[tag][layer_idx]['adj_means'][c] for layer_idx in layer_ids}
#         for tag in fold_tags
#     }
#     adj_class_minus_A0_by_fold_layer = {
#         tag: {layer_idx: test_stats_by_fold[tag][layer_idx]['adj_means_minus_A0'][c] for layer_idx in layer_ids}
#         for tag in fold_tags
#     }
#     alearn_class_by_fold_layer = {
#         tag: {layer_idx: test_stats_by_fold[tag][layer_idx]['alearn_means'][c] for layer_idx in layer_ids}
#         for tag in fold_tags
#     }
#     # plot_adj_fold_x_layer_grid(
#     #     adj_class_by_fold_layer,
#     #     layer_ids,
#     #     fold_tags,
#     #     save_path=TEST_ADJ_OUTPUT_DIR / f'adjacency_mean_{CLASS_NAMES[c]}_all_folds_all_layers.png',
#     #     suptitle=f'Test adjacency mean ({CLASS_NAMES[c]}) | folds × layers',
#     # )
#     plot_adj_fold_x_layer_grid(
#         adj_class_minus_A0_by_fold_layer,
#         layer_ids,
#         fold_tags,
#         save_path=TEST_ADJ_OUTPUT_DIR / f'adjacency_mean_{CLASS_NAMES[c]}_all_folds_all_layers_minus_A0.png',
#         suptitle=f'Test adjacency mean − A0 ({CLASS_NAMES[c]}) | folds × layers',
#         **MINUS_A0_PLOT_KW,
#     )
#     # plot_adj_fold_x_layer_grid(
#     #     alearn_class_by_fold_layer,
#     #     layer_ids,
#     #     fold_tags,
#     #     save_path=TEST_ADJ_OUTPUT_DIR / f'A_learn_mean_{CLASS_NAMES[c]}_all_folds_all_layers.png',
#     #     suptitle=f'Test A_learn mean ({CLASS_NAMES[c]}) | folds × layers',
#     # )

# # 五折汇总：折数×层数，每折单独 colorbar
# plot_adj_fold_x_layer_grid(
#     adj_overall_by_fold_layer,
#     layer_ids,
#     fold_tags,
#     save_path=TEST_ADJ_OUTPUT_DIR / 'adjacency_overall_mean_all_folds_all_layers.png',
#     suptitle='Test adjacency overall mean | folds × layers',
# )
# plot_adj_fold_x_layer_grid(
#     adj_overall_minus_A0_by_fold_layer,
#     layer_ids,
#     fold_tags,
#     save_path=TEST_ADJ_OUTPUT_DIR / 'adjacency_overall_mean_all_folds_all_layers_minus_A0.png',
#     suptitle='Test adjacency overall mean − A0 | folds × layers',
#     **MINUS_A0_PLOT_KW,
# )
# plot_adj_fold_x_layer_grid(
#     alearn_overall_by_fold_layer,
#     layer_ids,
#     fold_tags,
#     save_path=TEST_ADJ_OUTPUT_DIR / 'A_learn_overall_mean_all_folds_all_layers.png',
#     suptitle='Test A_learn overall mean | folds × layers',
# )

print(f"\n全部折测试集可视化完成: {TEST_ADJ_OUTPUT_DIR}")
print(f"共 {len(layer_ids)} 层 × {len(fold_tags)} 折")


itr_000 | seed=39
train ids: [2, 5, 6, 7, 8, 11, 13, 14, 15, 17, 19, 20, 22, 27, 29, 31, 32, 33, 34, 35, 36, 38, 39, 41, 42, 44, 46, 48, 49, 50, 52, 54, 55, 56, 57, 59, 60, 61, 64, 67, 69, 70, 74, 75, 77, 78, 79, 80, 81, 82, 84, 88, 90, 91, 92, 93, 96, 97, 98, 99, 100, 101, 102, 104, 105, 106, 107, 108, 109, 111] (数量: 70)
test ids: [1, 3, 4, 9, 10, 12, 18, 23, 25, 26, 40, 43, 45, 58, 63, 71, 76, 83, 85, 87, 89, 103, 110] (数量: 23)
collect_test_adjacency: 保留正确分类样本 2012/4089
collect_test_adjacency: 类别均衡采样 -> 每类 n=188 (NC: 1272->188, MCI: 552->188, AD: 188->188)
  layer 0 | NC: n=188
  layer 0 | MCI: n=188
  layer 0 | AD: n=188
  layer 1 | NC: n=188
  layer 1 | MCI: n=188
  layer 1 | AD: n=188
  layer 2 | NC: n=188
  layer 2 | MCI: n=188
  layer 2 | AD: n=188
  layer 3 | NC: n=188
  layer 3 | MCI: n=188
  layer 3 | AD: n=188
  layer 4 | NC: n=188
  layer 4 | MCI: n=188
  layer 4 | AD: n=188
  layer 5 | NC: n=188
  layer 5 | MCI: n=188
  layer 5 | AD: n=188
  已保存至 /root/autodl-tmp/ADSST-Ne

## Pearson 连通矩阵 组间差异性分析

In [23]:
# ========== 参数：选择要分析的折 ==========
# FOLD_IDX 与文件名 test_adj_stats_itr_XXX.npz 中的 XXX 一致：
#   FOLD_IDX=1 -> itr_001（第一折），FOLD_IDX=2 -> itr_002，…，FOLD_IDX=0 -> itr_000
FOLD_IDX = 1

# 数据目录（与上文 TEST_ADJ_OUTPUT_DIR 一致；若单独运行本节，则回退到 PROJECT_ROOT 下默认路径）
ADJ_STATS_DIR = (
    TEST_ADJ_OUTPUT_DIR
    if 'TEST_ADJ_OUTPUT_DIR' in globals()
    else PROJECT_ROOT / 'figs' / 'Weight Visualization Analysis/PearsonGraphConvolution/Adj_matrix_A_learn'
)
CLASS_NAMES = globals().get('CLASS_NAMES', ['NC', 'MCI', 'AD'])
CLASS_IDS = list(range(len(CLASS_NAMES)))


def fold_tag_from_idx(fold_idx: int) -> str:
    """fold_idx -> itr_XXX 标签（与保存 npz 时的命名一致）。"""
    return f'itr_{fold_idx:03d}'


def list_adj_stats_npz(data_dir: Path) -> list[Path]:
    """列出目录下所有 test_adj_stats_itr_*.npz 文件（按折号排序）。"""
    files = sorted(data_dir.glob('test_adj_stats_itr_*.npz'))
    return files


def _npz_to_python(value):
    """将 npz 中的 0 维 object / 字符串数组还原为 Python 对象。"""
    if isinstance(value, np.ndarray) and value.ndim == 0:
        return value.item()
    if isinstance(value, np.ndarray) and value.dtype.kind in {'U', 'S', 'O'}:
        return value.tolist() if value.ndim > 0 else str(value.item())
    return value


def load_test_adj_stats_npz(
    data_dir: Path,
    fold_idx: int,
) -> dict:
    """加载单折 test_adj_stats npz，并整理为便于后续组间差异分析的结构。

    返回字典主要字段
    ----------------
    meta : ch_names, layer_ids, fold_tag, fold_seed, A0, npz_path
    raw  : adj_by_layer, alearn_by_layer（逐样本矩阵，layer -> list[np.ndarray]）
    counts : {class_id: n_samples}
    layer_stats : {layer_idx: adj_means, adj_means_minus_A0, alearn_means,
                   adj_overall, adj_overall_minus_A0, alearn_overall}
    """
    fold_tag = fold_tag_from_idx(fold_idx)
    npz_path = data_dir / f'test_adj_stats_{fold_tag}.npz'
    if not npz_path.exists():
        available = [p.stem.replace('test_adj_stats_', '') for p in list_adj_stats_npz(data_dir)]
        raise FileNotFoundError(
            f'未找到 {npz_path}\n'
            f'可用折: {available or "(目录为空)"}'
        )

    with np.load(npz_path, allow_pickle=True) as raw:
        ch_names = list(_npz_to_python(raw['ch_names']))
        layer_ids = [int(x) for x in np.asarray(raw['layer_ids']).tolist()]
        fold_tag_loaded = str(_npz_to_python(raw['fold_tag']))
        fold_seed = int(_npz_to_python(raw['fold_seed']))
        A0 = np.asarray(raw['A0'], dtype=np.float64)
        adj_by_layer = _npz_to_python(raw['adj'])
        alearn_by_layer = _npz_to_python(raw['alearn'])

        counts = {
            c: int(_npz_to_python(raw[f'count_{CLASS_NAMES[c]}']))
            for c in CLASS_IDS
        }

        layer_stats = {}
        for layer_idx in layer_ids:
            adj_means = {
                c: np.asarray(raw[f'adj_mean_{CLASS_NAMES[c]}_layer{layer_idx}'], dtype=np.float64)
                for c in CLASS_IDS
            }
            adj_means_minus_A0 = {
                c: np.asarray(
                    raw[f'adj_mean_{CLASS_NAMES[c]}_layer{layer_idx}_minus_A0'],
                    dtype=np.float64,
                )
                for c in CLASS_IDS
            }
            alearn_means = {
                c: np.asarray(
                    raw[f'A_learn_mean_{CLASS_NAMES[c]}_layer{layer_idx}'],
                    dtype=np.float64,
                )
                for c in CLASS_IDS
            }
            layer_stats[layer_idx] = {
                'adj_means': adj_means,
                'adj_means_minus_A0': adj_means_minus_A0,
                'alearn_means': alearn_means,
                'adj_overall': np.asarray(
                    raw[f'adj_overall_mean_layer{layer_idx}'], dtype=np.float64,
                ),
                'adj_overall_minus_A0': np.asarray(
                    raw[f'adj_overall_mean_layer{layer_idx}_minus_A0'], dtype=np.float64,
                ),
                'alearn_overall': np.asarray(
                    raw[f'A_learn_overall_mean_layer{layer_idx}'], dtype=np.float64,
                ),
            }

    if fold_tag_loaded != fold_tag:
        print(f'[warn] 文件名折号={fold_tag}，文件内 fold_tag={fold_tag_loaded}')

    return {
        'meta': {
            'ch_names': ch_names,
            'layer_ids': layer_ids,
            'fold_tag': fold_tag_loaded,
            'fold_idx': fold_idx,
            'fold_seed': fold_seed,
            'A0': A0,
            'npz_path': npz_path,
        },
        'raw': {
            'adj_by_layer': adj_by_layer,
            'alearn_by_layer': alearn_by_layer,
        },
        'counts': counts,
        'layer_stats': layer_stats,
    }


# ---------- 加载当前折 ----------
available_npz = list_adj_stats_npz(ADJ_STATS_DIR)
print(f'ADJ_STATS_DIR = {ADJ_STATS_DIR}')
print(f'可用 npz ({len(available_npz)}): '
      f'{[p.stem.replace("test_adj_stats_", "") for p in available_npz]}')
print(f'当前选择: FOLD_IDX={FOLD_IDX} -> {fold_tag_from_idx(FOLD_IDX)}')

adj_stats = load_test_adj_stats_npz(ADJ_STATS_DIR, FOLD_IDX)
meta = adj_stats['meta']
layer_ids = meta['layer_ids']
ch_names = meta['ch_names']

print(
    f"\n已加载: {meta['npz_path'].name}\n"
    f"  fold_tag={meta['fold_tag']}, seed={meta['fold_seed']}, "
    f"layers={layer_ids}, n_ch={len(ch_names)}"
)
count_str = ', '.join(f'{CLASS_NAMES[c]}={adj_stats["counts"][c]}' for c in CLASS_IDS)
print(f'  各类样本数: {{{count_str}}}')
for layer_idx in layer_ids:
    sample_n = len(adj_stats['raw']['adj_by_layer'][layer_idx])
    mat_shape = adj_stats['layer_stats'][layer_idx]['adj_means'][CLASS_IDS[0]].shape
    print(f"  layer {layer_idx}: n_samples={sample_n}, matrix shape={mat_shape}")


ADJ_STATS_DIR = /root/autodl-tmp/ADSST-Net/figs/Prior-Guided Graph Learning/Adj_matrix_A_learn
可用 npz (5): ['itr_000', 'itr_001', 'itr_002', 'itr_003', 'itr_004']
当前选择: FOLD_IDX=1 -> itr_001

已加载: test_adj_stats_itr_001.npz
  fold_tag=itr_001, seed=40, layers=[0, 1, 2, 3, 4, 5], n_ch=20
  各类样本数: {NC=114, MCI=114, AD=114}
  layer 0: n_samples=342, matrix shape=(20, 20)
  layer 1: n_samples=342, matrix shape=(20, 20)
  layer 2: n_samples=342, matrix shape=(20, 20)
  layer 3: n_samples=342, matrix shape=(20, 20)
  layer 4: n_samples=342, matrix shape=(20, 20)
  layer 5: n_samples=342, matrix shape=(20, 20)


### 计算 不同人群 均值矩阵 差值，并绘制热力图

In [24]:
# ========== 不同人群 adj−A0 均值矩阵 组间差值 & 热力图 ==========
# 数据来源：adj_stats['layer_stats'][layer]['adj_means_minus_A0']
# （各类别 test adjacency 均值减 A0，与 adj_overall_minus_A0 同属 minus-A0 系列）

GROUP_DIFF_PAIRS = [
    (0, 1),  # NC − MCI
    (0, 2),  # NC − AD
    (1, 2),  # MCI − AD
]
GROUP_DIFF_LAYERS = None   # None = 全部层；或指定如 [0, 3, 5]

GROUP_DIFF_OUTPUT_DIR = (
    save_path / 'Group_Diff_adj_minus_A0'
    if 'save_path' in globals()
    else PROJECT_ROOT / 'figs' / 'Weight Visualization Analysis/PearsonGraphConvolution/Group_Diff_adj_minus_A0'
)
GROUP_DIFF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_diff_plot_base = MINUS_A0_PLOT_KW if 'MINUS_A0_PLOT_KW' in globals() else dict(
    cmap_name='RdBu_r', diverging=True,
)
DIFF_PLOT_KW = {**_diff_plot_base, 'value_label': 'Δ mean (adj − A0)'}


def _pair_label(class_a: int, class_b: int) -> str:
    return f'{CLASS_NAMES[class_a]} − {CLASS_NAMES[class_b]}'


def compute_group_mean_diff(
    layer_stats: dict,
    class_a: int,
    class_b: int,
    layer_idx: int,
    stat_key: str = 'adj_means_minus_A0',
) -> np.ndarray:
    """计算两类 adj−A0 均值矩阵之差：class_a − class_b。"""
    mat_a = np.asarray(layer_stats[layer_idx][stat_key][class_a], dtype=np.float64)
    mat_b = np.asarray(layer_stats[layer_idx][stat_key][class_b], dtype=np.float64)
    return mat_a - mat_b


def build_group_diff_by_pair_layer(
    adj_stats: dict,
    pairs: list[tuple[int, int]],
    layer_ids: list[int],
    stat_key: str = 'adj_means_minus_A0',
) -> dict[tuple[int, int], dict[int, np.ndarray]]:
    layer_stats = adj_stats['layer_stats']
    return {
        (c_a, c_b): {
            layer_idx: compute_group_mean_diff(layer_stats, c_a, c_b, layer_idx, stat_key)
            for layer_idx in layer_ids
        }
        for c_a, c_b in pairs
    }


def plot_group_diff_heatmaps_by_layer(
    diff_by_pair_layer: dict[tuple[int, int], dict[int, np.ndarray]],
    pairs: list[tuple[int, int]],
    layer_ids: list[int],
    fold_tag: str,
    output_dir: Path,
):
    """每层一张图：各组间差值并排展示（调用 plot_A0_heatmap_grid）。"""
    saved = []
    for layer_idx in layer_ids:
        mats = [diff_by_pair_layer[(c_a, c_b)][layer_idx] for c_a, c_b in pairs]
        labels = [_pair_label(c_a, c_b) for c_a, c_b in pairs]
        save_path = output_dir / f'group_diff_adj_minus_A0_{fold_tag}_layer{layer_idx}.png'
        plot_A0_heatmap_grid(
            mats,
            labels,
            save_path,
            suptitle=f'Group mean diff (adj − A0) | {fold_tag} | layer {layer_idx}',
            **DIFF_PLOT_KW,
        )
        saved.append(save_path)
    return saved


def plot_group_diff_heatmaps_by_pair(
    diff_by_pair_layer: dict[tuple[int, int], dict[int, np.ndarray]],
    pairs: list[tuple[int, int]],
    layer_ids: list[int],
    fold_tag: str,
    output_dir: Path,
):
    """每个组间对比一张图：各 encoder 层并排展示（调用 plot_adj_fold_x_layer_grid）。"""
    saved = []
    for c_a, c_b in pairs:
        pair_tag = f'{CLASS_NAMES[c_a]}_minus_{CLASS_NAMES[c_b]}'
        diff_by_layer = diff_by_pair_layer[(c_a, c_b)]
        data_by_fold = {fold_tag: diff_by_layer}
        save_path = output_dir / f'group_diff_adj_minus_A0_{pair_tag}_{fold_tag}_all_layers.png'
        plot_adj_fold_x_layer_grid(
            data_by_fold,
            layer_ids,
            [fold_tag],
            save_path,
            suptitle=f'Group mean diff (adj − A0): {_pair_label(c_a, c_b)} | {fold_tag}',
            **DIFF_PLOT_KW,
        )
        saved.append(save_path)
    return saved


# ---------- 计算 & 绘图 ----------
target_layers = layer_ids if GROUP_DIFF_LAYERS is None else list(GROUP_DIFF_LAYERS)
fold_tag = meta['fold_tag']

group_diff_by_pair_layer = build_group_diff_by_pair_layer(
    adj_stats, GROUP_DIFF_PAIRS, target_layers,
)

print(f'GROUP_DIFF_OUTPUT_DIR = {GROUP_DIFF_OUTPUT_DIR}')
print(f'fold={fold_tag}, layers={target_layers}, pairs={[ _pair_label(a, b) for a, b in GROUP_DIFF_PAIRS ]}')

for c_a, c_b in GROUP_DIFF_PAIRS:
    for layer_idx in target_layers:
        diff = group_diff_by_pair_layer[(c_a, c_b)][layer_idx]
        finite = diff[np.isfinite(diff)]
        print(
            f'  {_pair_label(c_a, c_b)} | layer {layer_idx}: '
            f'mean={finite.mean():+.6f}, std={finite.std():.6f}, '
            f'range=[{finite.min():+.6f}, {finite.max():+.6f}]'
        )

saved_by_layer = plot_group_diff_heatmaps_by_layer(
    group_diff_by_pair_layer, GROUP_DIFF_PAIRS, target_layers, fold_tag, GROUP_DIFF_OUTPUT_DIR,
)
saved_by_pair = plot_group_diff_heatmaps_by_pair(
    group_diff_by_pair_layer, GROUP_DIFF_PAIRS, target_layers, fold_tag, GROUP_DIFF_OUTPUT_DIR,
)

print(f'\n已保存 {len(saved_by_layer)} 张「按层」热力图、{len(saved_by_pair)} 张「按组间对比」热力图')
print(f'输出目录: {GROUP_DIFF_OUTPUT_DIR}')


GROUP_DIFF_OUTPUT_DIR = /root/autodl-tmp/ADSST-Net/figs/Prior-Guided Graph Learning/Group_Diff_adj_minus_A0
fold=itr_001, layers=[0, 1, 2, 3, 4, 5], pairs=['NC − MCI', 'NC − AD', 'MCI − AD']
  NC − MCI | layer 0: mean=-0.000000, std=0.002990, range=[-0.033121, +0.025524]
  NC − MCI | layer 1: mean=-0.000000, std=0.003450, range=[-0.030508, +0.024288]
  NC − MCI | layer 2: mean=-0.000000, std=0.005527, range=[-0.043357, +0.033745]
  NC − MCI | layer 3: mean=-0.000000, std=0.002552, range=[-0.026891, +0.026850]
  NC − MCI | layer 4: mean=-0.000000, std=0.004341, range=[-0.029498, +0.036266]
  NC − MCI | layer 5: mean=-0.000000, std=0.010321, range=[-0.141008, +0.142403]
  NC − AD | layer 0: mean=+0.000000, std=0.002986, range=[-0.024649, +0.025034]
  NC − AD | layer 1: mean=-0.000000, std=0.003807, range=[-0.031810, +0.032423]
  NC − AD | layer 2: mean=-0.000000, std=0.005269, range=[-0.027306, +0.026710]
  NC − AD | layer 3: mean=-0.000000, std=0.003513, range=[-0.041094, +0.041116]
  N

### 计算不同人群 adj−A0 均值矩阵组间变化，并绘制脑网络图

组间变化定义：`num − den`，即 `MCI−NC`、`AD−NC`、`AD−MCI`。

In [25]:
# ========== 不同人群 adj−A0 均值矩阵 组间变化 & 顶视 connectome ==========
# 数据来源：test_stats_by_fold[fold][layer]['adj_means_minus_A0']
# 组间变化：num − den，即 MCI−NC、AD−NC、AD−MCI
# 依赖上文：plot_connectome_grid、test_stats_by_fold、fold_tags、layer_ids

import pandas as pd

# (被减数类别, 减数类别)
GROUP_CHANGE_SPECS = [
    (1, 0),  # MCI − NC
    (2, 0),  # AD − NC
    (2, 1),  # AD − MCI
]
GROUP_CHANGE_LAYERS = None   # None = 全部层；或指定如 [0, 3, 5]

GROUP_CHANGE_OUTPUT_DIR = (
    save_path / 'Group_Change_adj_minus_A0_connectome'
    if 'save_path' in globals()
    else PROJECT_ROOT / 'figs' / 'Weight Visualization Analysis/PearsonGraphConvolution/Group_Change_adj_minus_A0_connectome'
)
GROUP_CHANGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_diff_conn_plot_base = DIFF_PLOT_KW if 'DIFF_PLOT_KW' in globals() else dict(
    cmap_name='RdBu_r', diverging=True, value_label='Δ mean (adj − A0)',
)
CHANGE_CONNECTOME_PLOT_KW = {
    **_diff_conn_plot_base,
    'value_label': 'Δ mean (adj − A0)',
    'linewidth': 1.5,
    'dpi': 600,
    'symmetrize': True,
}


def _gd_change_label(num_class: int, den_class: int) -> str:
    return f'{CLASS_NAMES[num_class]} − {CLASS_NAMES[den_class]}'


def _gd_change_tag(num_class: int, den_class: int) -> str:
    return f'{CLASS_NAMES[num_class]}_minus_{CLASS_NAMES[den_class]}'


def _gd_change_sheet_name(num_class: int, den_class: int) -> str:
    return _gd_change_tag(num_class, den_class)[:31]


def _matrix_node_labels(n_nodes: int) -> list[str]:
    if 'CH_NAMES' in globals() and n_nodes == len(CH_NAMES):
        return list(CH_NAMES)
    return [f'node{i}' for i in range(n_nodes)]


def compute_group_change_matrix(
    layer_stats: dict,
    num_class: int,
    den_class: int,
    layer_idx: int,
    stat_key: str = 'adj_means_minus_A0',
) -> np.ndarray:
    """计算组间变化矩阵：num − den。"""
    mat_num = np.asarray(layer_stats[layer_idx][stat_key][num_class], dtype=np.float64)
    mat_den = np.asarray(layer_stats[layer_idx][stat_key][den_class], dtype=np.float64)
    return mat_num - mat_den


def build_group_change_by_spec_layer(
    adj_stats: dict,
    specs: list[tuple[int, int]],
    layer_ids: list[int],
    stat_key: str = 'adj_means_minus_A0',
) -> dict[tuple[int, int], dict[int, np.ndarray]]:
    layer_stats = adj_stats['layer_stats']
    return {
        (num_c, den_c): {
            layer_idx: compute_group_change_matrix(
                layer_stats, num_c, den_c, layer_idx, stat_key,
            )
            for layer_idx in layer_ids
        }
        for num_c, den_c in specs
    }


def write_group_change_to_excel(
    change_by_spec_layer: dict[tuple[int, int], dict[int, np.ndarray]],
    specs: list[tuple[int, int]],
    layer_ids: list[int],
    fold_tag: str,
    xlsx_path: Path,
):
    """每个组间变化一张工作表，工作表内按 layer 分块写入矩阵。"""
    with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
        for num_c, den_c in specs:
            sheet_name = _gd_change_sheet_name(num_c, den_c)
            startrow = 0
            for layer_idx in layer_ids:
                mat = change_by_spec_layer[(num_c, den_c)][layer_idx]
                node_labels = _matrix_node_labels(mat.shape[0])
                title_df = pd.DataFrame([[
                    f'{fold_tag} | layer {layer_idx} | {_gd_change_label(num_c, den_c)}',
                ]])
                title_df.to_excel(
                    writer, sheet_name=sheet_name, startrow=startrow, index=False, header=False,
                )
                df = pd.DataFrame(mat, index=node_labels, columns=node_labels)
                df.to_excel(writer, sheet_name=sheet_name, startrow=startrow + 1)
                startrow += len(df) + 3
    return xlsx_path


def plot_group_change_connectomes_by_layer(
    change_by_spec_layer: dict[tuple[int, int], dict[int, np.ndarray]],
    specs: list[tuple[int, int]],
    layer_ids: list[int],
    fold_tag: str,
    output_dir: Path,
    **plot_kw,
):
    """每层一张图：各组间变化顶视 connectome 并排展示（每子图独立 colorbar）。"""
    if 'plot_connectome_grid' not in globals():
        raise NameError('请先运行定义 plot_connectome_grid 的 cell')
    saved = []
    for layer_idx in layer_ids:
        mats = [change_by_spec_layer[(num_c, den_c)][layer_idx] for num_c, den_c in specs]
        labels = [_gd_change_label(num_c, den_c) for num_c, den_c in specs]
        save_path = output_dir / f'group_change_adj_minus_A0_{fold_tag}_layer{layer_idx}_connectome'
        plot_connectome_grid(
            mats,
            labels,
            save_path,
            suptitle=f'Group change connectome (adj − A0) | {fold_tag} | layer {layer_idx}',
            **plot_kw,
            shared_colorbar=False,
        )
        saved.append(save_path.with_suffix('.png'))
    return saved


def plot_group_change_connectomes_by_spec(
    change_by_spec_layer: dict[tuple[int, int], dict[int, np.ndarray]],
    specs: list[tuple[int, int]],
    layer_ids: list[int],
    fold_tag: str,
    output_dir: Path,
    **plot_kw,
):
    """每个组间变化一张图：各 encoder 层顶视 connectome 并排展示（每子图独立 colorbar）。"""
    if 'plot_connectome_grid' not in globals():
        raise NameError('请先运行定义 plot_connectome_grid 的 cell')
    saved = []
    for num_c, den_c in specs:
        spec_tag = _gd_change_tag(num_c, den_c)
        mats = [change_by_spec_layer[(num_c, den_c)][layer_idx] for layer_idx in layer_ids]
        labels = [f'Layer {layer_idx}' for layer_idx in layer_ids]
        save_path = output_dir / f'group_change_adj_minus_A0_{spec_tag}_{fold_tag}_all_layers_connectome'
        plot_connectome_grid(
            mats,
            labels,
            save_path,
            suptitle=(
                f'Group change connectome (adj − A0): '
                f'{_gd_change_label(num_c, den_c)} | {fold_tag}'
            ),
            **plot_kw,
            shared_colorbar=False,
        )
        saved.append(save_path.with_suffix('.png'))
    return saved


# ---------- 计算、导出 Excel & 绘图 ----------
if 'test_stats_by_fold' not in globals() or 'fold_tags' not in globals():
    raise NameError('请先运行计算 test_stats_by_fold 的 cell')

specs = GROUP_CHANGE_SPECS
target_layers = layer_ids if GROUP_CHANGE_LAYERS is None else list(GROUP_CHANGE_LAYERS)
if GROUP_CHANGE_LAYERS is None and 'GROUP_DIFF_LAYERS' in globals() and GROUP_DIFF_LAYERS is not None:
    target_layers = list(GROUP_DIFF_LAYERS)

print(f'GROUP_CHANGE_OUTPUT_DIR = {GROUP_CHANGE_OUTPUT_DIR}')
print(f'layers={target_layers}, specs={[ _gd_change_label(n, d) for n, d in specs ]}')

for fold_tag in fold_tags:
    adj_stats = {'layer_stats': test_stats_by_fold[fold_tag]}
    group_change_by_spec_layer = build_group_change_by_spec_layer(
        adj_stats, specs, target_layers,
    )

    change_xlsx = GROUP_CHANGE_OUTPUT_DIR / f'group_change_adj_minus_A0_{fold_tag}.xlsx'
    write_group_change_to_excel(
        group_change_by_spec_layer, specs, target_layers, fold_tag, change_xlsx,
    )

    print(f'\nfold={fold_tag}')
    for num_c, den_c in specs:
        for layer_idx in target_layers:
            change = group_change_by_spec_layer[(num_c, den_c)][layer_idx]
            finite = change[np.isfinite(change)]
            if finite.size == 0:
                print(f'  {_gd_change_label(num_c, den_c)} | layer {layer_idx}: no finite values')
                continue
            print(
                f'  {_gd_change_label(num_c, den_c)} | layer {layer_idx}: '
                f'mean={finite.mean():+.6f}, std={finite.std():.6f}, '
                f'range=[{finite.min():+.6f}, {finite.max():+.6f}]'
            )

    saved_conn_by_layer = plot_group_change_connectomes_by_layer(
        group_change_by_spec_layer, specs, target_layers, fold_tag, GROUP_CHANGE_OUTPUT_DIR,
        **CHANGE_CONNECTOME_PLOT_KW,
    )
    saved_conn_by_spec = plot_group_change_connectomes_by_spec(
        group_change_by_spec_layer, specs, target_layers, fold_tag, GROUP_CHANGE_OUTPUT_DIR,
        **CHANGE_CONNECTOME_PLOT_KW,
    )
    print(f'已保存 Excel: {change_xlsx}')
    print(f'已保存 {len(saved_conn_by_layer)} 张「按层」connectome、{len(saved_conn_by_spec)} 张「按组间变化」connectome')

print(f'\n输出目录: {GROUP_CHANGE_OUTPUT_DIR}')


GROUP_CHANGE_OUTPUT_DIR = /root/autodl-tmp/ADSST-Net/figs/Prior-Guided Graph Learning/Group_Change_adj_minus_A0_connectome
layers=[0, 1, 2, 3, 4, 5], specs=['MCI − NC', 'AD − NC', 'AD − MCI']

fold=itr_000
  MCI − NC | layer 0: mean=+0.000000, std=0.002631, range=[-0.023890, +0.022616]
  MCI − NC | layer 1: mean=+0.000000, std=0.000958, range=[-0.013308, +0.013309]
  MCI − NC | layer 2: mean=-0.000000, std=0.000382, range=[-0.004475, +0.004441]
  MCI − NC | layer 3: mean=-0.000000, std=0.005858, range=[-0.055814, +0.046677]
  MCI − NC | layer 4: mean=-0.000000, std=0.000256, range=[-0.003330, +0.003300]
  MCI − NC | layer 5: mean=+0.000000, std=0.000638, range=[-0.008748, +0.008739]
  AD − NC | layer 0: mean=+0.000000, std=0.003005, range=[-0.024995, +0.025175]
  AD − NC | layer 1: mean=+0.000000, std=0.000079, range=[-0.001072, +0.001110]
  AD − NC | layer 2: mean=-0.000000, std=0.001284, range=[-0.017797, +0.017771]
  AD − NC | layer 3: mean=-0.000000, std=0.012577, range=[-0.148485,